# Blunder Prediction Under Time Pressure

This notebook turns the Stockfish blunder predictor into a readable ML case study. The production implementation lives in `ml.blunder_predictor`; this notebook loads the generated artifacts by default and keeps retraining as an optional reproducibility step.

**Question:** can time, board-state, and player-context features rank moves that are likely to become 200 centipawn Stockfish blunders?


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

try:
    from IPython.display import Image, Markdown, display
except ImportError:
    class Markdown(str):
        pass

    class Image:
        def __init__(self, filename: str):
            self.filename = filename

        def __repr__(self) -> str:
            return f"Image({self.filename})"

    def display(value) -> None:
        print(value)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "models").exists():
            return candidate
    return current.parent if current.name == "notebooks" else current


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def read_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def require_artifacts(paths: list[Path]) -> None:
    missing = [path for path in paths if not path.exists()]
    if missing:
        display(Markdown("### Missing artifacts"))
        display(Markdown("Run the matching `make train-*` target, then rerun this notebook."))
        for path in missing:
            display(Markdown(f"- `{path.relative_to(REPO_ROOT)}`"))
        raise FileNotFoundError("Required ML artifacts are missing")


def metric_frame(metrics: dict[str, float]) -> pd.DataFrame:
    return (
        pd.Series(metrics, name="value")
        .rename_axis("metric")
        .reset_index()
        .assign(value=lambda frame: frame["value"].map(lambda value: f"{float(value):.4f}"))
    )


def display_markdown_file(path: Path) -> None:
    display(Markdown(path.read_text(encoding="utf-8")))


print(f"Repo root: {REPO_ROOT}")

ARTIFACT_DIR = REPO_ROOT / "models" / "blunder_predictor"
required = [
    ARTIFACT_DIR / "metrics.json",
    ARTIFACT_DIR / "feature_importance.csv",
    ARTIFACT_DIR / "precision_recall_curve.png",
    ARTIFACT_DIR / "roc_curve.png",
    ARTIFACT_DIR / "threshold_tradeoff.png",
    ARTIFACT_DIR / "evaluation_report.md",
    ARTIFACT_DIR / "model_card.md",
]
require_artifacts(required)
metadata = read_json(ARTIFACT_DIR / "metrics.json")
metadata.keys()


## Dataset And Target

The training frame comes from `analytics.blunder_positions` in the benchmark DuckDB warehouse. Each row is a Stockfish-evaluated position sampled from the chess pipeline. The binary target is `is_blunder`, defined as a move with at least 200 centipawns of loss.

This is a rare-event problem, so raw accuracy is misleading: a model can score high accuracy by predicting "not blunder" for almost every move. The useful questions are ranking quality, recall, and the precision/recall tradeoff.


In [ ]:
class_counts = {int(label): count for label, count in metadata["class_counts"].items()}
summary = pd.DataFrame(
    [
        ("Rows", metadata["row_count"]),
        ("Training rows", metadata["train_rows"]),
        ("Test rows", metadata["test_rows"]),
        ("Non-blunders", class_counts.get(0, 0)),
        ("200cp blunders", class_counts.get(1, 0)),
        ("Positive rate", class_counts.get(1, 0) / metadata["row_count"]),
        ("Encoded features", metadata["features"]["encoded_count"]),
    ],
    columns=["item", "value"],
)
summary


## Feature Set

The model intentionally uses features available from the chess analytics pipeline plus Stockfish-labeled positions. Numeric features describe player Elo, clock pressure, ply number, material balance, check status, and year. Categorical features describe phase, time control, and destination square.


In [ ]:
features = metadata["features"]
pd.DataFrame(
    [("numeric", feature) for feature in features["numeric"]]
    + [("categorical", feature) for feature in features["categorical"]],
    columns=["feature_type", "feature"],
)


## Model Vs Baselines

The main model is an XGBoost binary classifier with class imbalance handled through `scale_pos_weight`. Baselines show whether the model beats simple rules.


In [ ]:
rows = [{"model": "xgboost", **metadata["metrics"]}]
for baseline_name, baseline_metrics in metadata["baselines"].items():
    rows.append({"model": baseline_name, **baseline_metrics})
comparison = pd.DataFrame(rows).set_index("model")
comparison[["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]].style.format("{:.4f}")


## Threshold Tradeoff

The default threshold is 0.50, but operationally this model is more useful as a risk-ranking tool. A chess coach or analyst might prefer higher recall to catch more candidate blunders, then review the shortlist manually.


In [ ]:
thresholds = pd.read_csv(ARTIFACT_DIR / "threshold_metrics.csv")
thresholds.head(10)


In [ ]:
display(Image(filename=str(ARTIFACT_DIR / "precision_recall_curve.png")))
display(Image(filename=str(ARTIFACT_DIR / "roc_curve.png")))
display(Image(filename=str(ARTIFACT_DIR / "threshold_tradeoff.png")))


## Feature Importance

Feature importance helps explain what the boosted trees used most often. Treat this as model interpretation, not causal proof.


In [ ]:
importance = pd.read_csv(ARTIFACT_DIR / "feature_importance.csv")
importance.head(20)


In [ ]:
display(Image(filename=str(ARTIFACT_DIR / "feature_importance.png")))


## Report And Model Card

The generated report and model card are the durable artifacts for review. They include the training configuration, metrics, artifacts, and limitations.


In [ ]:
display_markdown_file(ARTIFACT_DIR / "evaluation_report.md")


In [ ]:
display_markdown_file(ARTIFACT_DIR / "model_card.md")


## Interpretation

The model improves ROC-AUC and PR-AUC over simple baselines, but precision remains low because true 200cp blunders are rare in the evaluated sample. That is acceptable for a screening workflow, not for automated judgment. The right product framing is: "rank risky time-pressure moves for review," not "perfectly identify every blunder."


## Optional Retraining

Set `RUN_TRAINING = True` only when the benchmark DuckDB warehouse exists and you want to regenerate artifacts. The Python module remains the source of truth.


In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    from ml.blunder_predictor.train import TrainingConfig, train

    metadata = train(
        TrainingConfig(
            duckdb_path=REPO_ROOT / "warehouse" / "knightvision_benchmark.duckdb",
            output_dir=ARTIFACT_DIR,
        )
    )
    metadata["metrics"]
else:
    print("Skipping training. Set RUN_TRAINING = True to regenerate artifacts.")
